## 🤝 8. Human in the loop

**Goal**: Human in the loop

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import asyncio
import os
import random
from collections.abc import AsyncIterable
from dataclasses import dataclass, field

from agent_framework import (
    Agent,
    AgentExecutorRequest,
    AgentExecutorResponse,
    AgentResponse,
    AgentResponseUpdate,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    handler,
    response_handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from typing_extensions import Never

from utils.config import get_foundry_client, get_env

client = get_foundry_client()
print("✅ Connected to Foundry")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-26 11:48:37,407 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Connected to Foundry


---
#### 1. Create coordinator node for the worklow and handles the approval / reject provided by the human.

In [6]:
@dataclass
class DraftFeedbackRequest:
    """Payload sent for human review."""

    prompt: str = ""
    conversation: list[Message] = field(default_factory=lambda: [])

class Coordinator(Executor):
    """Bridge between the writer agent, human feedback, and final editor."""

    def __init__(self, id: str, writer_name: str, final_editor_name: str) -> None:
        super().__init__(id)
        self.writer_name = writer_name
        self.final_editor_name = final_editor_name

    @handler
    async def on_writer_response(
        self,
        draft: AgentExecutorResponse,
        ctx: WorkflowContext[Never, AgentResponse],
    ) -> None:
        """Handle responses from the writer and final editor agents."""
        if draft.executor_id == self.final_editor_name:
            # No further processing is needed when the final editor has responded.
            return

        # Writer agent response; request human feedback.
        # Preserve the full conversation so that the final editor has context.
        conversation = list(draft.full_conversation)

        prompt = (
            "Review the draft from the writer and provide a short directional note "
            "(tone tweaks, must-have detail, target audience, etc.). "
            "Keep it under 30 words."
        )
        await ctx.request_info(
            request_data=DraftFeedbackRequest(prompt=prompt, conversation=conversation),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: DraftFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """Process human feedback and forward to the appropriate agent."""
        note = feedback.strip()
        if note.lower() == "approve":
            # Human approved the draft as-is; forward it unchanged.
            await ctx.send_message(
                AgentExecutorRequest(
                    messages=original_request.conversation
                    + [Message("user", contents=["The draft is approved as-is."])],
                    should_respond=True,
                ),
                target_id=self.final_editor_name,
            )
            return

        # Human provided feedback; prompt the writer to revise.
        conversation: list[Message] = list(original_request.conversation)
        instruction = (
            "A human reviewer shared the following guidance:\n"
            f"{note or 'No specific guidance provided.'}\n\n"
            "Rewrite the draft from the previous assistant message into a polished final version. "
            "Keep the response under 120 words and reflect any requested tone adjustments."
        )
        conversation.append(Message("user", contents=[instruction]))
        await ctx.send_message(
            AgentExecutorRequest(messages=conversation, should_respond=True),
            target_id=self.writer_name,
        )


async def process_event_stream(stream: AsyncIterable[WorkflowEvent]) -> dict[str, str] | None:
    """Process events from the workflow stream to capture feedback requests.

    In notebook mode, this function defaults to random auto-feedback for demo runs.
    """
    # Track the last author to format streaming output.
    last_author: str | None = None

    requests: list[tuple[str, DraftFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, DraftFeedbackRequest):
            requests.append((event.request_id, event.data))
        elif event.type == "output" and isinstance(event.data, AgentResponseUpdate):
            # This workflow should only produce AgentResponseUpdate as outputs.
            # Streaming updates from an agent will be consecutive, because no two agents run simultaneously
            # in this workflow. So we can use last_author to format output nicely.
            update = event.data
            author = update.author_name
            if author != last_author:
                if last_author is not None:
                    print()  # Newline between different authors
                print(f"{author}: {update.text}", end="", flush=True)
                last_author = author
            else:
                print(update.text, end="", flush=True)

    # Handle pending human feedback requests.
    if requests:
        responses: dict[str, str] = {}
        for request_id, _ in requests:
            print("\nProvide guidance for the editor (or 'approve' to accept the draft).")            
            # answer = input("Human feedback: ").strip()  # noqa: ASYNC250
            answer = random.choices(
                ["approve", "rejected. Please rewrite"],
                weights=[0.1, 0.8],
                k=1,
            )[0]
            print(f"Human feedback (auto-random): {answer}")
            if answer.lower() == "exit":
                print("Exiting...")
                return None
            responses[request_id] = answer
        return responses

    return None

print("✅ Coordinator node defined")

✅ Coordinator node defined


---
#### 1. Create workflow with coordinator, writer agent and final editor agent.

In [7]:
# Create the agents
writer_agent = Agent(
    client=client,
    name="writer_agent",
    instructions=("You are a marketing writer."),
    default_options={
        "tool_choice": "required",
    },
)

final_editor_agent = Agent(
    client=client,
    name="final_editor_agent",
    instructions=(
        "You are an editor who polishes marketing copy after human approval. "
        "Correct any legal or factual issues. Return the final version even if no changes are made. "
    ),
)

# Create the executor
coordinator = Coordinator(
    id="coordinator",
    writer_name=writer_agent.name,  # type: ignore
    final_editor_name=final_editor_agent.name,  # type: ignore
)

# Build the workflow.
workflow = (
    WorkflowBuilder(
        start_executor=writer_agent,
        output_from="all",
    )
    .add_edge(writer_agent, coordinator)
    .add_edge(coordinator, writer_agent)
    .add_edge(final_editor_agent, coordinator)
    .add_edge(coordinator, final_editor_agent)
    .build()
)

In [8]:
print(
    "Notebook mode: pending human feedback requests are auto-approved.",
    flush=True,
)

# Initiate the first run of the workflow.
# Runs are not isolated; state is preserved across multiple calls to run.
stream = workflow.run(
    "Create a short launch blurb for the LumenX desk lamp. Emphasize adjustability and warm lighting.",
    stream=True,
)

pending_responses = await process_event_stream(stream)
while pending_responses is not None:
    # Run the workflow until there is no more feedback to provide,
    # in which case this workflow completes.
    stream = workflow.run(stream=True, responses=pending_responses)
    pending_responses = await process_event_stream(stream)

print("\nWorkflow complete.")

Notebook mode: pending human feedback requests are auto-approved.
writer_agent: Introducing the LumenX Desk Lamp: Illuminate your workspace with style and precision. Designed for ultimate adjustability, the LumenX easily adapts to your needs, while its advanced warm lighting technology creates the perfect ambiance for productivity and comfort. Transform your desk into a haven of efficiency and elegance with the LumenX—where innovation meets illumination.
Provide guidance for the editor (or 'approve' to accept the draft).
Human feedback (auto-random): rejected. Please rewrite
writer_agent: Illuminate your workspace with the new LumenX Desk Lamp. Featuring unparalleled adjustability, the LumenX allows you to tailor your lighting to any task or mood. Its warm lighting technology offers a soothing ambiance that enhances both productivity and comfort. Bring elegance and efficiency to your desk with the LumenX—where versatility meets serenity.
Provide guidance for the editor (or 'approve' to